To generate the table, run propeller_lookup_table.py. Once generated successfully, the data will be stored in lookup_table folder. The following section is an example to generate a subset of a lookup table. For debug and development purpose, generating a small lookup table as trial can save time. 

In [ ]:
import numpy as np
from propeller_lookup_table import PropellerLookupTable
import blade_params

PropellerLookupTable.Maker.make_propeller_lookup_table("apc_8x6_simple", blade_params.APC_8x6(), np.array(PropellerLookupTable.Maker._DEFAULT_OMEGA_RANGE[5:]), np.array(
    PropellerLookupTable.Maker._DEFAULT_U_FREE_X_RANGE[8:9]), np.array(PropellerLookupTable.Maker._DEFAULT_PITCH_RANGE[5:]))

The following example generates the full lookup table of apc_8x6.

In [ ]:
from propeller_lookup_table import PropellerLookupTable

PropellerLookupTable.Maker.make_propeller_lookup_table("apc_8x6_with_trail_refine")

The following example generates the full lookup table of apc_8x6 with fitted parameters.

In [ ]:
from propeller_lookup_table import PropellerLookupTable
import blade_params

blade = blade_params.APC_8x6()
blade.cl_1, blade.cl_2, blade.cd, blade.alpha_0 = (5.31194203, 1.46627105, 1.72517079, 0.45174155)  # fitted parameters
blade.cl_1, blade.cl_2, blade.cd, blade.alpha_0 = (5.19938603, 1.34972972, 1.74778588, 0.45789112)  # fitted parameters
PropellerLookupTable.Maker.make_propeller_lookup_table("apc_8x6_fitted2", blade,)

The following example generates lookup table for P600 paper blade params.

In [ ]:
import numpy as np

from propeller_lookup_table import PropellerLookupTable
import blade_params

is_simple_table = False   
if is_simple_table:
    PropellerLookupTable.Maker.make_propeller_lookup_table("p600", blade_params.P600_Blade(), 
                                                        np.array(PropellerLookupTable.Maker._DEFAULT_OMEGA_RANGE[:10]), 
                                                        np.array(PropellerLookupTable.Maker._DEFAULT_U_FREE_X_RANGE[0:2]), 
                                                        np.array(PropellerLookupTable.Maker._DEFAULT_PITCH_RANGE[4:7]))
else:
    PropellerLookupTable.Maker.make_propeller_lookup_table("p600_full_range", blade_params.P600_Blade(),
                                                        np.array(PropellerLookupTable.Maker._DEFAULT_OMEGA_RANGE[:10]), 
                                                        np.array(PropellerLookupTable.Maker._DEFAULT_U_FREE_X_RANGE), 
                                                        np.array(PropellerLookupTable.Maker._DEFAULT_PITCH_RANGE))

The following example generates lookup table for NeuroBEM paper blade params.

In [ ]:
from propeller_lookup_table import PropellerLookupTable
import blade_params

PropellerLookupTable.Maker.make_propeller_lookup_table("neurobem", blade_params.Neurobem())

To use the lookup table, users can find the table data under lookup_table. The following example shows how to load the lookup table and query the rotor forces.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from propeller_lookup_table import PropellerLookupTable
import blade_params

def compute_forces_grouped_by_pitch_then_omega(reader, u_free_x_range, pitch_range, omega_range):
    forces_grouped_by_pitch_then_omega = []
    for u_free_x in u_free_x_range:
        for pitch in pitch_range:
            forces_by_omega = []
            for omega in omega_range:
                forces_by_omega.append(reader.query_data_from_table(u_free_x, pitch, omega))
            forces_grouped_by_pitch_then_omega.append(forces_by_omega)
    forces_grouped_by_pitch_then_omega = np.array(forces_grouped_by_pitch_then_omega)
    return forces_grouped_by_pitch_then_omega

def compute_forces_sensed_wind(reader, u_free_x_range, pitch_range, omega_range):
    forces = []
    for u_free_x in u_free_x_range:
        for pitch in pitch_range:
            forces_by_omega = []
            for omega in omega_range:
                v_i = reader.query_data_from_table(u_free_x, pitch, omega)[3]
                u_disk_plane = u_free_x * np.cos(pitch)
                u_normal = u_free_x * np.sin(pitch) - v_i
                forces_by_omega.append(reader.query_data_from_table_sensed_wind(u_disk_plane, u_normal, omega))
            forces.append(forces_by_omega)
    return np.array(forces)

def plot_forces(omega_range, pitch_range, forces_grouped_by_pitch_then_omega,
                comparison_data: blade_params.GroundTruthBladeData=None,
                axes=None, label_prefix=''):
    if axes is None:
        figs = [plt.figure() for _ in range(4)]
        axes = [fig.add_subplot(111) for fig in figs]

    for i, pitch in enumerate(pitch_range):
        axes[0].plot(omega_range, forces_grouped_by_pitch_then_omega[i, :, 0], label=fr"{label_prefix}$F_x$ Pitch Angle {np.rad2deg(pitch):.2f}")
    axes[0].set_xlabel("Omega (rad/s)")
    axes[0].set_ylabel("F (N)")
    axes[0].legend()
    axes[0].grid()

    for i, pitch in enumerate(pitch_range):
        axes[1].plot(omega_range, forces_grouped_by_pitch_then_omega[i, :, 1], label=fr"{label_prefix}$F_y$ Pitch Angle {np.rad2deg(pitch):.2f}")
    axes[1].set_xlabel("Omega (rad/s)")
    axes[1].set_ylabel("F (N)")
    axes[1].legend()
    axes[1].grid()

    for i, pitch in enumerate(pitch_range):
        axes[2].plot(omega_range, forces_grouped_by_pitch_then_omega[i, :, 2], label=fr"{label_prefix}$F_z$ Pitch Angle {np.rad2deg(pitch):.2f}")
    # optional comparison data (same position as original line)
    if comparison_data is not None:
        axes[2].plot(comparison_data.get_omega_range(), comparison_data.get_thrust_range(), 'o', label='Ground Truth')
    axes[2].set_xlabel("Omega (rad/s)")
    axes[2].set_ylabel("Thrust (N)")
    axes[2].legend()
    axes[2].grid()

    for i, pitch in enumerate(pitch_range):
        axes[3].plot(omega_range, forces_grouped_by_pitch_then_omega[i, :, 3], label=f"{label_prefix}vi Pitch {np.rad2deg(pitch):.2f}")
    axes[3].set_xlabel("Omega (rad/s)")
    axes[3].set_ylabel("trail wind (m/s)")
    axes[3].legend()
    axes[3].grid()

    return axes


reader = PropellerLookupTable.Reader("p600_full_range")

u_free_x_range = [0]
pitch_range = np.deg2rad([-90, -60, -30, 0, 30, 60, 90])
pitch_range = np.deg2rad([-60, 0, 60])    # for paper fig
omega_range = np.array([0, 200, 300, 400, 500, 600, 700, 800, 900, 1000])

forces_grouped_by_pitch_then_omega = compute_forces_grouped_by_pitch_then_omega(reader, u_free_x_range, pitch_range, omega_range)
axes = plot_forces(omega_range, pitch_range, forces_grouped_by_pitch_then_omega, blade_params.P600_SimData, label_prefix='free-stream ')

forces_sensed = compute_forces_sensed_wind(reader, u_free_x_range, pitch_range, omega_range)
plot_forces(omega_range, pitch_range, forces_sensed, axes=axes, label_prefix='sensed-wind ')

plt.show()


The following example shows how to compute rotor forces.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from propeller_lookup_table import PropellerLookupTable

apc_8x6 = PropellerLookupTable.Reader("apc_8x6_with_trail")
u_free = np.array([10, 0, 0])
v_forward = np.array([0, 0, 0])
omega = 800
r_disk = np.eye(3)
is_ccw = True

print("fig 1: rotor straight up")
forces, v_i = apc_8x6.get_rotor_forces(u_free, v_forward, r_disk, omega, is_ccw)
PropellerLookupTable.Reader.plot_rotor_force(r_disk, forces, v_i)

print("fig 2: rotor frame rotated in x-y plane (should not change the force)")
theta = np.pi / 3
r_disk = np.array([[np.cos(theta), np.sin(theta), 0],   # x axis after transpose
                   [-np.sin(theta), np.cos(theta), 0],  # y axis after transpose
                   [0, 0, 1]]).T    # z axis after transpose
forces, v_i = apc_8x6.get_rotor_forces(u_free, v_forward, r_disk, omega, is_ccw)
PropellerLookupTable.Reader.plot_rotor_force(r_disk, forces, v_i)


print("fig 3: forward velocity (should be the same as fig 2)")
v_forward = np.array([-10, 0, 0])
u_free = np.array([0, 0, 0])
forces, v_i = apc_8x6.get_rotor_forces(u_free, v_forward, r_disk, omega, is_ccw)
PropellerLookupTable.Reader.plot_rotor_force(r_disk, forces, v_i)

print("fig 4: z axis along the free stream")
r_disk = np.array([[0, 1, 0],   # x axis after transpose
                   [0, 0, 1],   # y axis after transpose
                   [1, 0, 0]]).T    # z axis after transpose
forces, v_i = apc_8x6.get_rotor_forces(u_free, v_forward, r_disk, omega, is_ccw)
PropellerLookupTable.Reader.plot_rotor_force(r_disk, forces, v_i)

plt.show()
